In [3]:
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from tqdm import tqdm
import os

# ======================== Enhanced Core Components with SBERT ========================

# 1. Data Preparation & SRL Processing
class SRLProcessor:
    def __init__(self):
        self.role_tags = {
            'V': 'verb', 'ARG0': 'subject', 'ARG1': 'object',
            'ARG2': 'indirect-object', 'ARGM-MNR': 'manner'
        }
    
    def process_srl(self, srl_data):
        """Process SRL data and add semantic role labels to text"""
        augmented = []
        for sent_info in srl_data:
            if 'srl_raw' not in sent_info or 'words' not in sent_info['srl_raw']:
                continue  # Skip malformed entries
                
            words = sent_info['srl_raw']['words']
            tags = [[] for _ in words]
            
            # Process each verb and its tags
            for verb_info in sent_info['srl_raw'].get('verbs', []):
                if 'tags' not in verb_info:
                    continue
                    
                current_tags = verb_info['tags']
                for idx, tag in enumerate(current_tags):
                    if idx >= len(tags):  # Prevent index error
                        break
                    if tag != 'O' and '-' in tag:
                        role = tag.split('-')[1]
                        if role in self.role_tags:  # Only add if it's in our defined roles
                            tags[idx].append(role)
            
            # Build the tagged sentence
            tagged_sentence = []
            for word, roles in zip(words, tags):
                for role in roles:
                    if role in self.role_tags:
                        tagged_sentence.append(f"[{self.role_tags[role]}]")
                tagged_sentence.append(word)
                for role in reversed(roles):
                    if role in self.role_tags:
                        tagged_sentence.append(f"[/{self.role_tags[role]}]")
            
            augmented.append(' '.join(tagged_sentence))
        
        return ' '.join(augmented)

def load_srl_dataset(json_path):
    """Load dataset from JSON and apply SRL processing"""
    try:
        with open(json_path) as f:
            data = json.load(f).get('samples', [])
    except (json.JSONDecodeError, FileNotFoundError) as e:
        print(f"Error loading JSON data: {e}")
        return pd.DataFrame()
    
    processor = SRLProcessor()
    samples = []
    
    for sample in tqdm(data, desc="Processing SRL data", leave=False):
        try:
            samples.append({
                'CVE_text': processor.process_srl(sample.get('CVE_srl', [])),
                'Technique_text': processor.process_srl(sample.get('Technique_srl', [])),
                'label': sample.get('label', 0),
                'role_score': sample.get('role_match_score', 0.0)
            })
        except Exception as e:
            print(f"Error processing sample: {e}")
            continue
    
    df = pd.DataFrame(samples)
    print(f"Loaded {len(df)} valid samples")
    return df

# 2. Dataset Class with Hinge Labels - Using Text Instead of Tokens for SBERT
class HingeSBERTDataset(Dataset):
    def __init__(self, df):
        self.df = df
        
        # Normalize role weights to [0,1]
        if len(df) > 0:  # Check if df is not empty
            role_min = df['role_score'].min()
            role_max = df['role_score'].max()
            self.role_weights = (df['role_score'] - role_min) / (role_max - role_min + 1e-8)
            
            # Convert to -1/1 labels for hinge loss
            self.labels = 2 * df['label'].values - 1
        else:
            self.role_weights = pd.Series()
            self.labels = np.array([])

    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # For SBERT, we don't tokenize here - we just return the text
        return {
            'cve_text': row['CVE_text'],
            'tech_text': row['Technique_text'],
            'labels': torch.tensor(self.labels[idx], dtype=torch.float),
            'role_weights': torch.tensor(self.role_weights.iloc[idx], dtype=torch.float)
        }

# 3. Enhanced Model Architecture with SBERT and Contrastive Learning
class SBERTContrastiveModel(nn.Module):
    def __init__(self, model_name="all-mpnet-base-v2", hidden_size=768, margin=0.4, contrastive_weight=0.3):
        super().__init__()
        try:
            # Use SentenceTransformer for better sentence embeddings
            self.sbert = SentenceTransformer(model_name)
            self.hidden_size = self.sbert.get_sentence_embedding_dimension()
        except Exception as e:
            print(f"Error loading pretrained SBERT model: {e}")
            raise RuntimeError("Failed to initialize the model")
        
        # Projection layer for embedding fine-tuning
        self.srl_proj = nn.Sequential(
            nn.Linear(self.hidden_size, 256),
            nn.ReLU(),
            nn.LayerNorm(256)
        )
        
        # Classifier using aggregation of embeddings and interaction features
        self.classifier = nn.Sequential(
            nn.Linear(256 * 4, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 1)
        )
        
        self.contrastive_loss = nn.CosineEmbeddingLoss(margin=margin)
        self.contrastive_weight = contrastive_weight

    def forward(self, cve_texts=None, tech_texts=None, labels=None, cve_emb=None, tech_emb=None):
        # If embeddings are not provided, compute them
        if cve_emb is None and tech_emb is None:
            with torch.no_grad():
                # Get embeddings from SBERT
                cve_emb = torch.tensor(self.sbert.encode(cve_texts, convert_to_numpy=True), 
                                       dtype=torch.float).to(next(self.parameters()).device)
                tech_emb = torch.tensor(self.sbert.encode(tech_texts, convert_to_numpy=True), 
                                        dtype=torch.float).to(next(self.parameters()).device)
        
        # Project embeddings to a lower dimension
        cve_proj = self.srl_proj(cve_emb)
        tech_proj = self.srl_proj(tech_emb)
        
        # Aggregating features from both branches
        diff = torch.abs(cve_proj - tech_proj)
        prod = cve_proj * tech_proj
        combined_features = torch.cat([cve_proj, tech_proj, diff, prod], dim=1)  # [B, 256*4]
        
        # Compute classifier output (similarity score or decision)
        classifier_out = self.classifier(combined_features).squeeze()
        
        # In case labels is None (inference mode)
        cont_loss = torch.tensor(0.0, device=classifier_out.device)
        
        # Optionally, compute contrastive loss if labels provided
        if labels is not None:
            contrastive_labels = torch.where(labels > 0, 1.0, -1.0).to(labels.device)
            cont_loss = self.contrastive_loss(cve_proj, tech_proj, contrastive_labels)
        
        # Always return both values to maintain consistent return structure
        return classifier_out, cont_loss

    def get_embeddings(self, texts):
        """Get embeddings for inference"""
        with torch.no_grad():
            # Get SBERT embeddings
            embeddings = self.sbert.encode(texts, convert_to_numpy=True)
            # Project embeddings
            embeddings_tensor = torch.tensor(embeddings, dtype=torch.float)
            projected = self.srl_proj(embeddings_tensor)
            return projected.numpy()

# 4. Enhanced Hinge Loss with Weighting (unchanged)
class WeightedHingeLoss(nn.Module):
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin
        
    def forward(self, outputs, labels, weights):
        losses = torch.clamp(self.margin - labels * outputs, min=0)
        return (losses * (1 + weights)).mean()

# 5. Data Splitting with Stratification (unchanged)
def get_splits_for_model(df, test_size=0.15, val_size=0.15, random_state=42):
    """Split data into train/val/test with stratification"""
    if df.empty:
        raise ValueError("DataFrame is empty, cannot split")
        
    # First split off the test set
    train_val_df, test_df = train_test_split(
        df, test_size=test_size, random_state=random_state, stratify=df['label']
    )
    
    # Then split the remaining data into train and validation
    relative_val_size = val_size / (1 - test_size)
    train_df, val_df = train_test_split(
        train_val_df, 
        test_size=relative_val_size, 
        random_state=random_state,
        stratify=train_val_df['label']
    )
    
    print(f"Train: {len(train_df)}, Validation: {len(val_df)}, Test: {len(test_df)}")
    print(f"Train label distribution: {train_df['label'].value_counts().to_dict()}")
    print(f"Val label distribution: {val_df['label'].value_counts().to_dict()}")
    print(f"Test label distribution: {test_df['label'].value_counts().to_dict()}")
    
    return train_df, val_df, test_df

# 6. Enhanced Training with SBERT and Contrastive Loss
def train_sbert_model(train_df, val_df=None, epochs=10, batch_size=16, 
                     margin=1.0, patience=3, lr=2e-5, contrastive_weight=0.4,
                     sbert_model="all-mpnet-base-v2"):
    """Train model with contrastive learning objective using SBERT embeddings"""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    # Use the SBERT model
    model = SBERTContrastiveModel(model_name=sbert_model, contrastive_weight=contrastive_weight).to(device)
    
    train_dataset = HingeSBERTDataset(train_df)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    if val_df is not None:
        val_dataset = HingeSBERTDataset(val_df)
        val_loader = DataLoader(val_dataset, batch_size=batch_size)
    else:
        val_loader = None
    
    optimizer = optim.AdamW(model.parameters(), lr=lr)
    criterion = WeightedHingeLoss(margin=margin)
    
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc = 0
    patience_counter = 0
    
    for epoch in range(epochs):
        model.train()
        train_loss = cont_loss_total = 0
        train_correct = train_total = 0
        
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} - Training", leave=True):
            # Get batch data
            cve_texts = batch['cve_text']
            tech_texts = batch['tech_text']
            labels = batch['labels'].to(device)
            weights = batch['role_weights'].to(device)
            
            # Pre-compute embeddings (no grad needed for SBERT part)
            with torch.no_grad():
                cve_emb = torch.tensor(model.sbert.encode(cve_texts, convert_to_numpy=True), 
                                       dtype=torch.float).to(device)
                tech_emb = torch.tensor(model.sbert.encode(tech_texts, convert_to_numpy=True), 
                                        dtype=torch.float).to(device)
            
            optimizer.zero_grad()
            # Forward pass: model returns (classifier_out, contrastive_loss)
            outputs, cont_loss = model(cve_emb=cve_emb, tech_emb=tech_emb, labels=labels)
            
            hinge_loss = criterion(outputs, labels, weights)
            total_loss = hinge_loss + cont_loss * model.contrastive_weight
            
            total_loss.backward()
            optimizer.step()
            
            train_loss += total_loss.item()
            cont_loss_total += cont_loss.item()
            preds = torch.sign(outputs)
            train_correct += (preds == labels).sum().item()
            train_total += labels.size(0)
        
        avg_train_loss = train_loss / len(train_loader)
        avg_cont_loss = cont_loss_total / len(train_loader)
        train_acc = train_correct / train_total
        history['train_loss'].append(avg_train_loss)
        history['train_acc'].append(train_acc)
        
        print(f"Epoch {epoch+1}/{epochs} - Train Loss: {avg_train_loss:.4f}, Contrastive Loss: {avg_cont_loss:.4f}, Train Acc: {train_acc:.4f}")
        
        if val_loader is not None:
            val_acc, val_loss = validate_sbert_model(model, val_loader, criterion, device)
            history['val_loss'].append(val_loss)
            history['val_acc'].append(val_acc)
            
            print(f"Epoch {epoch+1}/{epochs} - Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
            
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                patience_counter = 0
                torch.save(model.state_dict(), "/kaggle/working/best_sbert_model.pth")
                print(f"Saved new best model with validation accuracy: {best_val_acc:.4f}")
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    print(f"Early stopping at epoch {epoch+1}")
                    model.load_state_dict(torch.load("/kaggle/working/best_sbert_model.pth"))
                    break
    
    if val_loader is None or patience_counter < patience:
        torch.save(model.state_dict(), "/kaggle/working/final_sbert_model.pth")
    
    plot_training_history(history)
    return model

def validate_sbert_model(model, val_loader, criterion, device):
    """Validate SBERT model on validation set"""
    model.eval()
    val_loss = val_correct = val_total = 0
    
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validation", leave=False):
            # Get batch data
            cve_texts = batch['cve_text']
            tech_texts = batch['tech_text']
            labels = batch['labels'].to(device)
            weights = batch['role_weights'].to(device)
            
            # Get embeddings
            cve_emb = torch.tensor(model.sbert.encode(cve_texts, convert_to_numpy=True), 
                                   dtype=torch.float).to(device)
            tech_emb = torch.tensor(model.sbert.encode(tech_texts, convert_to_numpy=True), 
                                    dtype=torch.float).to(device)
            
            # Forward pass
            outputs, cont_loss = model(cve_emb=cve_emb, tech_emb=tech_emb, labels=labels)
            
            # Calculate loss
            hinge_loss = criterion(outputs, labels, weights)
            total_loss = hinge_loss + cont_loss * model.contrastive_weight
            
            val_loss += total_loss.item()
            
            # Calculate accuracy
            preds = torch.sign(outputs)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)
    
    return val_correct/val_total, val_loss/len(val_loader)

# Plot training history (unchanged)
def plot_training_history(history):
    """Plot training and validation metrics"""
    plt.figure(figsize=(12, 5))
    
    # Plot loss
    plt.subplot(1, 2, 1)
    plt.plot(history['train_loss'], label='Train Loss')
    if 'val_loss' in history and history['val_loss']:
        plt.plot(history['val_loss'], label='Val Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.title('Loss Curves')
    
    # Plot accuracy
    plt.subplot(1, 2, 2)
    plt.plot(history['train_acc'], label='Train Accuracy')
    if 'val_acc' in history and history['val_acc']:
        plt.plot(history['val_acc'], label='Val Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.title('Accuracy Curves')
    
    plt.tight_layout()
    plt.savefig('/kaggle/working/training_history_sbert.png')
    plt.close()

# 8. Evaluation for SBERT model
def evaluate_sbert_model(model, test_df, device, batch_size=16):
    """Evaluate SBERT model and output detailed results on test set"""
    model.eval()
    test_dataset = HingeSBERTDataset(test_df)
    test_loader = DataLoader(test_dataset, batch_size=batch_size)
    
    # For metrics
    all_preds = []
    all_true = []
    all_outputs = []
    test_correct = test_total = 0
    
    # Test sample storage for detailed analysis
    test_samples = {
        'cve_text': [],
        'tech_text': [],
        'true_label': [],
        'predicted_label': [],
        'confidence_score': []
    }
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Evaluating on test set", leave=False):
            # Get batch data
            cve_texts = batch['cve_text']
            tech_texts = batch['tech_text']
            labels = batch['labels'].to(device)
            
            # Get embeddings
            cve_emb = torch.tensor(model.sbert.encode(cve_texts, convert_to_numpy=True), 
                                   dtype=torch.float).to(device)
            tech_emb = torch.tensor(model.sbert.encode(tech_texts, convert_to_numpy=True), 
                                    dtype=torch.float).to(device)
            
            # Forward pass
            outputs, _ = model(cve_emb=cve_emb, tech_emb=tech_emb, labels=labels)
            
            # Calculate accuracy
            preds = torch.sign(outputs)
            test_correct += (preds == labels).sum().item()
            test_total += labels.size(0)
            
            all_preds.extend(preds.cpu().numpy())
            all_true.extend(labels.cpu().numpy())
            all_outputs.extend(outputs.cpu().numpy())
            
            # Store samples for analysis
            for i in range(len(cve_texts)):
                test_samples['cve_text'].append(cve_texts[i])
                test_samples['tech_text'].append(tech_texts[i])
                test_samples['true_label'].append(float(labels[i].cpu().numpy()))
                test_samples['predicted_label'].append(float(preds[i].cpu().numpy()))
                test_samples['confidence_score'].append(float(outputs[i].cpu().numpy()))
    
    # Calculate metrics
    test_acc = test_correct / test_total
    print(f"Test Accuracy: {test_acc:.4f}")
    
    # Create a DataFrame for test results for easier analysis and output
    test_results_df = pd.DataFrame(test_samples)
    
    # Convert labels from -1/1 to 0/1 for readability
    test_results_df['true_label'] = (test_results_df['true_label'] + 1) / 2
    test_results_df['predicted_label'] = (test_results_df['predicted_label'] + 1) / 2
    
    # Add a column for correct/incorrect predictions
    test_results_df['correct'] = test_results_df['true_label'] == test_results_df['predicted_label']
    
    # Save detailed test results to CSV
    test_results_df.to_csv('/kaggle/working/test_results_detailed_sbert.csv', index=False)
    print(f"Saved detailed test results to 'test_results_detailed_sbert.csv'")
    
    # Classification report
    all_true_01 = [(label + 1) / 2 for label in all_true]  # Convert -1/1 to 0/1
    all_preds_01 = [(pred + 1) / 2 for pred in all_preds]  # Convert -1/1 to 0/1
    
    class_report = classification_report(all_true_01, all_preds_01, output_dict=True)
    print("\nClassification Report:")
    for label, metrics in class_report.items():
        if label in ['0.0', '1.0']:
            print(f"Class {label}: Precision: {metrics['precision']:.4f}, Recall: {metrics['recall']:.4f}, F1: {metrics['f1-score']:.4f}")
    
    # Confusion matrix
    cm = confusion_matrix(all_true_01, all_preds_01)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'])
    plt.title('Confusion Matrix (SBERT)')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.savefig('/kaggle/working/confusion_matrix_sbert.png')
    plt.close()
    
    # Sample analysis: Show some correct and incorrect examples
    print("\n=== Sample Correct Predictions ===")
    correct_samples = test_results_df[test_results_df['correct']].head(5)
    for i, row in correct_samples.iterrows():
        print(f"True label: {int(row['true_label'])}, Predicted: {int(row['predicted_label'])}, Confidence: {row['confidence_score']:.4f}")
        print(f"CVE excerpt: {row['cve_text'][:100]}...")
        print(f"Technique excerpt: {row['tech_text'][:100]}...")
        print("-" * 50)
    
    print("\n=== Sample Incorrect Predictions ===")
    incorrect_samples = test_results_df[~test_results_df['correct']].head(5)
    for i, row in incorrect_samples.iterrows():
        print(f"True label: {int(row['true_label'])}, Predicted: {int(row['predicted_label'])}, Confidence: {row['confidence_score']:.4f}")
        print(f"CVE excerpt: {row['cve_text'][:100]}...")
        print(f"Technique excerpt: {row['tech_text'][:100]}...")
        print("-" * 50)
    
    # Distribution of confidence scores
    plt.figure(figsize=(10, 6))
    correct_scores = test_results_df[test_results_df['correct']]['confidence_score']
    incorrect_scores = test_results_df[~test_results_df['correct']]['confidence_score']
    
    plt.hist(correct_scores, alpha=0.7, label='Correct predictions', bins=20)
    plt.hist(incorrect_scores, alpha=0.7, label='Incorrect predictions', bins=20)
    plt.title('Distribution of Model Confidence Scores (SBERT)')
    plt.xlabel('Confidence Score')
    plt.ylabel('Count')
    plt.legend()
    plt.savefig('/kaggle/working/confidence_distribution_sbert.png')
    plt.close()
    
    # Error analysis summary
    print("\n=== Error Analysis Summary ===")
    print(f"Total test samples: {len(test_results_df)}")
    print(f"Correct predictions: {len(test_results_df[test_results_df['correct']])} ({len(test_results_df[test_results_df['correct']])/len(test_results_df)*100:.2f}%)")
    print(f"Incorrect predictions: {len(test_results_df[~test_results_df['correct']])} ({len(test_results_df[~test_results_df['correct']])/len(test_results_df)*100:.2f}%)")
    
    # False positives and false negatives
    false_positives = test_results_df[(test_results_df['true_label'] == 0) & (test_results_df['predicted_label'] == 1)]
    false_negatives = test_results_df[(test_results_df['true_label'] == 1) & (test_results_df['predicted_label'] == 0)]
    
    print(f"False positives: {len(false_positives)} ({len(false_positives)/len(test_results_df)*100:.2f}%)")
    print(f"False negatives: {len(false_negatives)} ({len(false_negatives)/len(test_results_df)*100:.2f}%)")
    
    return test_acc, test_results_df

# SBERT Embeddings Visualization
def visualize_sbert_embeddings(model, test_df, device, max_samples=500):
    """Generate t-SNE visualization of SBERT embeddings"""
    print("Generating SBERT embedding visualizations...")
    try:
        # Sample a subset for visualization (t-SNE can be slow with large datasets)
        vis_sample = test_df.sample(min(max_samples, len(test_df))).reset_index(drop=True)
        
        # Extract embeddings
        cve_texts = vis_sample['CVE_text'].tolist()
        tech_texts = vis_sample['Technique_text'].tolist()
        labels = 2 * vis_sample['label'].values - 1  # Convert to -1/1
        
        model.eval()
        with torch.no_grad():
            # Get embeddings
            cve_embeddings = model.get_embeddings(cve_texts)
            tech_embeddings = model.get_embeddings(tech_texts)
        
        # Apply t-SNE for dimensionality reduction
        print("Applying t-SNE for visualization...")
        tsne = TSNE(n_components=2, random_state=42, perplexity=30)
        
        # Combine embeddings for visualization
        combined_embeddings = np.vstack([cve_embeddings, tech_embeddings])
        combined_labels = np.concatenate([labels, labels])
        
        # Add a type indicator (0 for CVE, 1 for Technique)
        types = np.concatenate([np.zeros(len(cve_embeddings)), np.ones(len(tech_embeddings))])
        
        # Apply t-SNE
        reduced_embeddings = tsne.fit_transform(combined_embeddings)
        
        # Plot
        plt.figure(figsize=(12, 10))
        
        # Convert to 0/1 label (from -1/+1)
        binary_labels = (combined_labels + 1) / 2
        
        # Create a scatter plot with four categories:
        # 1. CVE - Negative, 2. CVE - Positive, 3. Technique - Negative, 4. Technique - Positive
        markers = {0: 'o', 1: '^'}  # circle for CVE, triangle for Technique
        colors = {0: 'red', 1: 'blue'}  # red for negative, blue for positive
        
        for t in [0, 1]:  # type (CVE or Technique)
            for l in [0, 1]:  # label (0 or 1)
                mask = (types == t) & (binary_labels == l)
                plt.scatter(
                    reduced_embeddings[mask, 0],
                    reduced_embeddings[mask, 1],
                    marker=markers[t],
                    c=colors[l],
                    alpha=0.7,
                    label=f"{'CVE' if t == 0 else 'Technique'} - {'Negative' if l == 0 else 'Positive'}"
                )
        
        plt.legend()
        plt.title('t-SNE Visualization of CVE and Technique Embeddings (SBERT)')
        plt.savefig('/kaggle/working/embeddings_visualization_sbert.png')
        plt.close()
        print("Saved embeddings visualization to 'embeddings_visualization_sbert.png'")
        
    except Exception as e:
        print(f"Error generating visualizations: {e}")

# Main function
def main():
    """Main function to run the SBERT model training and evaluation pipeline"""
    # Set seed for reproducibility
    torch.manual_seed(42)
    np.random.seed(42)
    
    # Load data
    print("Loading SRL dataset...")
    try:
        df = load_srl_dataset("/kaggle/input/input1/siamese_samples_with_srl (6).json")
        if df.empty:
            print("Error: Dataset is empty. Please check the data file.")
            return
    except Exception as e:
        print(f"Error loading dataset: {e}")
        return
    
    # Split data
    try:
        train_df, val_df, test_df = get_splits_for_model(df)
    except Exception as e:
        print(f"Error splitting data: {e}")
        return
    
    # Choose SBERT model - can try different SBERT models
    sbert_model_name = "all-mpnet-base-v2"  # One of the best performing SBERT models
    print(f"Using SBERT model: {sbert_model_name}")
    
    # Check if we need to train or load a saved model
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # model_path assignment and remaining code for main() function
    model_path = "/kaggle/working/best_sbert_model.pth"
    
    if os.path.exists(model_path):
        print(f"Loading pre-trained model from {model_path}...")
        model = SBERTContrastiveModel(model_name=sbert_model_name).to(device)
        model.load_state_dict(torch.load(model_path, map_location=device))
    else:
        print("Training new model...")
        # Train model
        model = train_sbert_model(
            train_df=train_df,
            val_df=val_df,
            epochs=10,
            batch_size=16,
            margin=0.4,
            patience=3,
            lr=2e-5,
            contrastive_weight=0.3,
            sbert_model=sbert_model_name
        )
    
    # Evaluate model on test set
    print("\nEvaluating model on test set...")
    test_acc, test_results_df = evaluate_sbert_model(model, test_df, device)
    
    # Visualize embeddings
    print("\nVisualizing embeddings...")
    visualize_sbert_embeddings(model, test_df, device)
    
    print(f"\nFinal test accuracy: {test_acc:.4f}")
    print("Done!")

# Run the main function if script is executed directly
if __name__ == "__main__":
 main()


Loading SRL dataset...


Loaded 6759 valid samples
Train: 4731, Validation: 1014, Test: 1014
Train label distribution: {0: 3581, 1: 1150}
Val label distribution: {0: 767, 1: 247}
Test label distribution: {0: 767, 1: 247}
Using SBERT model: all-mpnet-base-v2
Training new model...
Using device: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Test Accuracy: 0.9329
Saved detailed test results to 'test_results_detailed_sbert.csv'

Classification Report:
Class 0.0: Precision: 0.9691, Recall: 0.9413, F1: 0.9550
Class 1.0: Precision: 0.8327, Recall: 0.9069, F1: 0.8682

=== Sample Correct Predictions ===
True label: 0, Predicted: 0, Confidence: -0.6288
CVE excerpt: [subject] Buffer [/subject] [subject] overflow [/subject] [subject] in [/subject] [subject] libtelne...
Technique excerpt: [subject] Adversaries [/subject] may [verb] attempt [/verb] to [verb] cause [/verb] [object] a [/obj...
--------------------------------------------------
True label: 1, Predicted: 1, Confidence: 0.8176
CVE excerpt: [indirect-object] In [/indirect-object] [indirect-object] Pulse [/indirect-object] [indirect-object]...
Technique excerpt: [subject] Adversaries [/subject] may [verb] leverage [/verb] [object] external [/object] [object] - ...
--------------------------------------------------
True label: 0, Predicted: 0, Confidence: -0.8326
CVE excerpt

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Error generating visualizations: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu! (when checking argument for argument mat1 in method wrapper_CUDA_addmm)

Final test accuracy: 0.9329
Done!


In [5]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score

# Load the detailed test results produced by your evaluation function
results_df = pd.read_csv('/kaggle/working/test_results_detailed_sbert.csv')

# Extract true labels and predicted labels as integer arrays
true_labels = results_df['true_label'].astype(int).values
pred_labels = results_df['predicted_label'].astype(int).values

# Calculate metrics
accuracy = accuracy_score(true_labels, pred_labels)
f1 = f1_score(true_labels, pred_labels)
recall = recall_score(true_labels, pred_labels)
precision = precision_score(true_labels, pred_labels)

print(f'Accuracy: {accuracy:.4f}')
print(f'F1 Score: {f1:.4f}')
print(f'Recall: {recall:.4f}')
print(f'Precision: {precision:.4f}')


Accuracy: 0.9329
F1 Score: 0.8682
Recall: 0.9069
Precision: 0.8327
